In [22]:
# Cell 1: Import all required libraries

import os
import numpy as np
from tqdm import tqdm

from skimage.io import imread
from skimage.transform import resize
from skimage.feature import hog

import torch
import torch.nn as nn
from torchvision import models, transforms


1) I chose the weights of Resnet50 to extract the features
2) I Removed the last layer (responsible for classification) and kept the backbone of resnet to receive only features
3) Set the model to evaluation mode cuz it's only needed for features extraction and not train
4) Last block will preprocess the images ( resize to Resnet expected resolution , normalize each RGB channel using the mean and deviation taken from Pytorch documentations) -> this gives better accuracy

In [ ]:
#Cell 2: Prepare device and load pretrained ResNet-50 backbon

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load ResNet50 (without the classification head)
resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
resnet.fc = nn.Identity()  # Remove the final fully connected layer. IMPORTANT!!!
resnet = resnet.to(device)
resnet.eval()  # Set model to evaluation mode because we need  it only for feature extraction 

# Preprocessing pipeline aligned with ImageNet requirements
preprocess = transforms.Compose([
    transforms.ToPILImage(), # Convert numpy array to pillow Image
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # ImageNet mean ( average) IMPORTANT!!!
        std=[0.229, 0.224, 0.225]    # ImageNet standard deviation IMPORTANT!!! 
    )
])


In [24]:
# Cell 3: Define feature extraction function for ResNet 

@torch.no_grad() # Just to save time
def extract_resnet_features(img):
    img_t = preprocess(img).unsqueeze(0).to(device)  # Apply preprocessing and add batch dimension
    features = resnet(img_t)                         # Forward pass through ResNet (without the FC layer)
    return features.squeeze().cpu().numpy()          # Remove batch dimension, move to CPU or gpu, convert to NumPy


In [25]:
# Cell 4: Define feature extraction function for HOG (edges/textures)

from skimage.color import rgb2gray

def extract_hog_features(img):
    img_gray = rgb2gray(img) # Convert to grayscale
    img_gray = resize(img_gray, (128, 64), preserve_range=True)

    features = hog(
        img_gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
    )
    return features


Function to load our data to our CPU or GPU which will:
1) read and sort the folders of our 6 classes
2) gives every class its label number
3) extract the features using Resnet and HOG in these classes and then concatenate them to only 1 vector then save them

In [26]:
# Cell 5: Helper to load a dataset folder and build combined feature vectors

def load_dataset(path):     # train or test folder path 
    X = []  # Features
    y = [] # Labels

    classes = sorted(os.listdir(path)) # sort randomly and read the path folders

    for label, cls in enumerate(classes):# enumerate(classes) turns it into pairs: (0, "buildings") for example
        folder = os.path.join(path, cls) # Create the full path to the current class folder
        
        for file in tqdm(os.listdir(folder), desc=f"Loading {cls}"):
            img_path = os.path.join(folder, file)

            img = imread(img_path) # Create the full path to the current image file.

            # ResNet Features
            f_res = extract_resnet_features(img) # extract resnet features IMPORTANT!!!

            # HOG Features
            f_hog = extract_hog_features(img) # extract hog features IMPORTANT!!!

            # Combine
            f_combined = np.concatenate([f_res, f_hog]) # Concatenate ResNet and HOG features into one long vector. # IMPORTANT!!!

            X.append(f_combined)
            y.append(label)

    return np.array(X), np.array(y)


In [27]:
# Cell 6: Load train/test sets and inspect feature dimensions

BASE_PATH = os.path.abspath(".")

TRAIN_DIR = os.path.join(BASE_PATH, "seg_train")
TEST_DIR  = os.path.join(BASE_PATH, "seg_test")

X_train, y_train = load_dataset(TRAIN_DIR) # IMPORTANT!!!
X_test,  y_test  = load_dataset(TEST_DIR) # IMPORTANT!!!

# ResNet handles color information 
# HOG focuses on edge and texture information
print("Train Shape:", X_train.shape) # links = number of trained images , feature vector dimension # IMPORTANT!!!
print("Test Shape:",  X_test.shape)


Loading street: 100%|██████████| 501/501 [00:06<00:00, 73.14it/s]

Train Shape: (14034, 5828)
Test Shape: (3000, 5828)


Here the hyperplane which seperates the features vectors is linear ( usually faster and safer) . Safer means less overfitting
here

In [ ]:
# Cell 7: Train Linear SVM and evaluate performance

from sklearn.metrics import accuracy_score
from sklearn.metrics import hinge_loss
import time
from sklearn.svm import SVC


svm = SVC(kernel="linear", probability=False) # probability : post-processing step called Platt Scaling ( doesn't say immediately the label , there is probability distribution)

# ---- training ----
start_train = time.time()
svm.fit(X_train, y_train)
end_train = time.time()

train_time = end_train - start_train
train_time_min = int(train_time // 60)
train_time_sec = train_time % 60

# ---- evaluation ----
y_pred = svm.predict(X_test)

# decision function outputs (needed for hinge loss)
scores = svm.decision_function(X_test)

# hinge loss 
svm_loss = hinge_loss(y_test, scores)

print("Test Accuracy:", accuracy_score(y_test, y_pred)) # IMPORTANT!!!
print(f"Training Time: {train_time_min} min {train_time_sec:.2f} sec") # IMPORTANT!!!
print(f"Test Loss: {svm_loss:.4f}")  # IMPORTANT!!!


Test Accuracy: 0.908
Training Time: 0 min 40.47 sec
Test Loss: 0.1935


Hyperplane  is non linear ( boundary is curved)

In [32]:
# Cell 8: Train RBF-SVM and report accuracy
from sklearn.metrics import accuracy_score, hinge_loss

from sklearn.svm import SVC

# RBF-SVM model
svm_rbf = SVC(
    kernel="rbf",
    C=1,
    gamma="scale",
    probability=False
)

# Training
start_train = time.time()
svm_rbf.fit(X_train, y_train)
end_train = time.time()

train_time = end_train - start_train
train_time_min = int(train_time // 60)
train_time_sec = train_time % 60

# Prediction
pred = svm_rbf.predict(X_test)

# Decision scores for hinge loss
scores_rbf = svm_rbf.decision_function(X_test)

# Hinge loss on test set
rbf_loss = hinge_loss(y_test, scores_rbf)

print("RBF-SVM Accuracy:", accuracy_score(y_test, pred))  # IMPORTANT!!!
print(f"Training Time: {train_time_min} min {train_time_sec:.2f} sec")  # IMPORTANT!!!
print(f"Test Loss (RBF): {rbf_loss:.4f}")



RBF-SVM Accuracy: 0.9306666666666666
Training Time: 1 min 2.63 sec
Test Loss (RBF): 0.1447


k-NN achieves good accuracy in this dataset because the HOG+CNN features form well-structured clusters.
However, the method remains unstable in high-dimensional spaces , can easily collapse and give bad accuracy.

-I don't trust this model because usually it's not very confident like in this result , as you see the loss is quite high 0.7

In [ ]:
# Cell 9: Train k-NN and report accuracy + training time

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, log_loss

# Define k-NN model
knn = KNeighborsClassifier(
    n_neighbors=5,       
    weights="uniform",   
    metric="minkowski"   # Standard
)

# Training
start_train = time.time()
knn.fit(X_train, y_train)
end_train = time.time()

train_time = end_train - start_train
train_time_min = int(train_time // 60)
train_time_sec = train_time % 60

# Prediction
# Prediction
pred_knn = knn.predict(X_test)

# Class probabilities for loss
proba_knn = knn.predict_proba(X_test)

# Log loss on test set
knn_loss = log_loss(y_test, proba_knn)

# Output
print("kNN Accuracy:", accuracy_score(y_test, pred_knn))
print(f"Training Time: {train_time_min} min {train_time_sec:.2f} sec")
print(f"Test Log Loss: {knn_loss:.4f}")



kNN Accuracy: 0.9206666666666666
Training Time: 0 min 0.06 sec
Test Log Loss: 0.7404


In [31]:
# Cell 10: Train Random Forest 

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,  # 300 trees
    max_depth=100,     # Limits tree depth
    n_jobs=-1,        # Cpu all cores used
    bootstrap=True     # More stable results and reduced overfitting
)

# Training
start_train = time.time()
rf.fit(X_train, y_train)
end_train = time.time()

train_time = end_train - start_train
train_time_min = int(train_time // 60)
train_time_sec = train_time % 60

# Prediction
pred_rf = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, pred_rf))
print(f"Training Time: {train_time_min} min {train_time_sec:.2f} sec")


Random Forest Accuracy: 0.917
Training Time: 0 min 37.36 sec
